Crossing Tagging - Riodades (v2)
=
Second version of *Tagueamento Cruzamentos*. It re-numbers the crossing tags of UFV Riodades per crossing category, and writes AutoCAD scripts that re-insert them as multileaders on one layer per category.

**Differences from v1:** the CSV is read from the project folder, and the tags go on layers named `EMF_Crossing_<category>` instead of `Crossing-<category>`. The layers script was not updated and still creates `Crossing-<category>`.

**Workflow overview:**
1. Read the existing crossing tags and their positions.
2. Sort each tag into one of 7 crossing categories by the cable/interference codes in its text.
   - Renumber the tags inside each category (top-to-bottom, left-to-right), and inside each crossing type for the two "secondary" categories.
3. Write two AutoCAD scripts: one that creates a layer per category and one that re-inserts every tag as an `MLEADER` on its category layer.

**Tag format:** `CR.<system>.<interference>-NNN` (e.g. `CR.LVAC.LVDC-001`, `CR.MV.DRN-012`).

**Input prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** `Riodades - interferências.csv` in `C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Cruzamentos\` (hardcoded), with **no header row** and exactly 4 columns in this order: X, Y, Z, tag text. Z is dropped.
- Every tag ends with a 3-character number (`-001`): the last 3 characters are replaced by the new number.
- The tag text is matched case-sensitively (`LVAC`, `LVDC`, `MV`, `DRN`).

**Outputs / AutoCAD:**
- `Riodades - create_layers.scr`: creates `Crossing-AC_DC`, `Crossing-AC_DRN`, ... These are **not** the layers the leaders script uses.
- `Riodades - reinsert_tags.scr`: sets each `EMF_Crossing_<category>` layer and draws the multileaders. The `EMF_Crossing_*` layers must already exist in the drawing (`-LAYER S` fails on a missing layer). It doesn't erase the old tags.
- Both files have the same names as the v1 outputs, so running this notebook overwrites them.

Classifying, renumbering and writing the scripts
=
- Reads the CSV (no header) as `Position X`, `Position Y`, `Position Z`, `TAG`, and drops Z.
- `categories` maps each category code to a description. Only the keys are used: they name the DataFrames and the layers.
- `allocate` appends one row to the category DataFrame, with `Layer` = `EMF_Crossing_<code>`.
- **Classification** (substring match on the tag, case-sensitive, first match wins, so the order matters):
  1. `LVAC.LVDC` → `AC_DC`
  2. `LVAC.DRN` → `AC_DRN`
  3. `LVDC.DRN` → `DC_DRN`
  4. `MV.DRN` → `MV_DRN`
  5. `MV.LVAC` → `MV_AC`
  6. any other tag containing `LVAC` → `AC_SEC`
  7. any other tag containing `MV` → `MV_SEC`
  - A tag that matches none of these (e.g. `CR.LVDC.WALL-001`) is dropped silently.
- `Type of Crossing` = the tag without the `CR.` prefix and the `-NNN` suffix (e.g. `LVAC.INTAC`). It is computed for the two "secondary" categories.
- **Numbering:**
  - Each category is sorted by `Position Y` descending, then `Position X` ascending (top-to-bottom, left-to-right).
  - The last 3 characters of each tag are replaced by the running number (`001`, `002`, ...).
  - `AC_SEC` and `MV_SEC` are then re-sorted by `Type of Crossing` and renumbered from `001` inside each type (e.g. `CR.LVAC.WALL-001`). This overwrites their first numbering, which is therefore redundant.
- **Leaders script:** all the crossings are sorted by layer. On each layer change it writes `-LAYER S <layer>` followed by a blank line, which ends the `-LAYER` command. The `current_layer is not None` check is always true (`current_layer` starts as `[]`), so the blank line is always written, which is what the command needs.
- Each crossing is one `MLEADER`: arrow at the crossing point, landing 10 units left and 5 up (`x-10, y+5`), text = the tag. The current multileader style is used.
- **Layers script:** `_-LAYER _M Crossing-<category>` for each of the 7 categories (Make creates the layer and sets it current). It is written after the leaders script, but in AutoCAD it must be run **first**.
- **Warning:** the leaders go on `EMF_Crossing_<category>`, but the layers script creates `Crossing-<category>`. Either create the `EMF_Crossing_*` layers by hand or fix the layer names.
- `time`/`start` and `numpy` are not used.

In [6]:
import time

import pandas as pd
import numpy as np

start = time.time()

column_names = ["Position X", "Position Y", "Position Z", "TAG"]

df_TAGs = pd.read_csv(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Cruzamentos\Riodades - interferências.csv",
                      header=None, names=column_names)

df_TAGs = df_TAGs.drop("Position Z", axis=1)

categories = {
    "AC_DC": "Crossing AC Cables with DC Cables",
    "AC_DRN": "Crossing AC Cables with Civil Drainage",
    "AC_SEC": "Crossing AC Cables with Secondary Interferences",
    "DC_DRN": "Crossing DC Cables with Civil Drainage",
    "MV_DRN": "Crossing MV Cables with Civil Drainage",
    "MV_AC": "Crossing MV Cables with AC Cables",
    "MV_SEC": "Crossing MV Cables with Secondary Interferences"
}

dfs = {
    name: pd.DataFrame(columns=["Position X", "Position Y", "TAG", "Layer"])
    for name in categories.keys()
}


def allocate(df_name, i):
    df = dfs[df_name]
    new_row = {"Position X": i[0], "Position Y": i[1], "TAG": i[2], "Layer": ("EMF_Crossing_"+df_name)}
    df.loc[len(df)] = new_row


def sort_by_xy_coordinates(df):
    return df.sort_values(by=["Position Y", "Position X"], ascending=[False, True]).reset_index(drop=True)


def tagging(df):
    return df["TAG"].astype(str).str[:-3] + (df.index+1).astype(str).str.zfill(3)


for item in df_TAGs.values:
    if "LVAC.LVDC" in item[2]:
        allocate("AC_DC", item)
        continue
    elif "LVAC.DRN" in item[2]:
        allocate("AC_DRN", item)
        continue
    elif "LVDC.DRN" in item[2]:
        allocate("DC_DRN", item)
        continue
    elif "MV.DRN" in item[2]:
        allocate("MV_DRN", item)
        continue
    elif "MV.LVAC" in item[2]:
        allocate("MV_AC", item)
        continue
    elif "LVAC" in item[2]:
        allocate("AC_SEC", item)
        continue
    elif "MV" in item[2]:
        allocate("MV_SEC", item)

dfs["MV_SEC"]["Type of Crossing"] = dfs["MV_SEC"]["TAG"].astype(str).str[3:-4]
dfs["AC_SEC"]["Type of Crossing"] = dfs["AC_SEC"]["TAG"].astype(str).str[3:-4]

for type_of_crossing in dfs:
    print(type_of_crossing)
    dfs[type_of_crossing] = sort_by_xy_coordinates(dfs[type_of_crossing])
    dfs[type_of_crossing]["TAG"] = tagging(dfs[type_of_crossing])


dfs["MV_SEC"] = dfs["MV_SEC"].sort_values(by=["Type of Crossing", "Position Y", "Position X"], ascending=[True, False, True])
groups = (dfs["MV_SEC"]["Type of Crossing"] != dfs["MV_SEC"]["Type of Crossing"].shift()).cumsum()
dfs["MV_SEC"]["INDEX"] = dfs["MV_SEC"].groupby(groups).cumcount() + 1
dfs["MV_SEC"]["TAG"] = dfs["MV_SEC"]["TAG"].astype(str).str[:-3] + (dfs["MV_SEC"]["INDEX"]).astype(str).str.zfill(3)

dfs["AC_SEC"] = dfs["AC_SEC"].sort_values(by=["Type of Crossing", "Position Y", "Position X"], ascending=[True, False, True])
groups = (dfs["AC_SEC"]["Type of Crossing"] != dfs["AC_SEC"]["Type of Crossing"].shift()).cumsum()
dfs["AC_SEC"]["INDEX"] = dfs["AC_SEC"].groupby(groups).cumcount() + 1
dfs["AC_SEC"]["TAG"] = dfs["AC_SEC"]["TAG"].astype(str).str[:-3] + (dfs["AC_SEC"]["INDEX"]).astype(str).str.zfill(3)

print(dfs["AC_SEC"])

scr_filename = r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Cruzamentos\Riodades - reinsert_tags.scr"

with open(scr_filename, 'w') as scr_file:
    current_layer = []
    all_leaders = []
    for df in dfs.values():
        for _, row in df.iterrows():
            all_leaders.append((
                row["Layer"],
                row["Position X"],
                row["Position Y"],
                row["TAG"]
            ))

    # Process leaders with correct command spacing
    for layer, x, y, tag in sorted(all_leaders, key=lambda x: x[0]):
        if layer != current_layer:
            scr_file.write(f"-LAYER S {layer}\n")
            if current_layer is not None:  # Only add newline if not first command
                scr_file.write("\n")
            current_layer = layer

        scr_file.write("MLEADER\n")
        scr_file.write(f"{x},{y}\n")
        scr_file.write(f"{x-10},{y+5}\n")
        scr_file.write(f"{tag}\n")
        # No extra newline here - critical!

    scr_file.write("\n")  # Final newline only at end

scr_filename = r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Cruzamentos\Riodades - create_layers.scr"

with open(scr_filename, 'w') as scr_file:
    for layer_name in categories.keys():
        scr_file.write(f"_-LAYER _M Crossing-{layer_name} \n")

AC_DC
AC_DRN
AC_SEC
DC_DRN
MV_DRN
MV_AC
MV_SEC
   Position X   Position Y                TAG                Layer  \
0  55338.0005  150563.6217  CR.LVAC.INTAC-001  EMF_Crossing_AC_SEC   
1  55313.6160  150563.4018  CR.LVAC.INTAC-002  EMF_Crossing_AC_SEC   
2  55590.6818  150555.1350  CR.LVAC.INTAC-003  EMF_Crossing_AC_SEC   
3  55585.2446  150553.4320  CR.LVAC.INTAC-004  EMF_Crossing_AC_SEC   
4  54854.2633  150425.5137  CR.LVAC.INTAC-005  EMF_Crossing_AC_SEC   
6  55258.9609  149928.1003  CR.LVAC.INTAC-006  EMF_Crossing_AC_SEC   
5  54864.3798  150271.6152   CR.LVAC.WALL-001  EMF_Crossing_AC_SEC   
7  55557.3030  149743.0928   CR.LVAC.WALL-002  EMF_Crossing_AC_SEC   
8  55587.0369  149725.8598   CR.LVAC.WALL-003  EMF_Crossing_AC_SEC   

  Type of Crossing  INDEX  
0       LVAC.INTAC      1  
1       LVAC.INTAC      2  
2       LVAC.INTAC      3  
3       LVAC.INTAC      4  
4       LVAC.INTAC      5  
6       LVAC.INTAC      6  
5        LVAC.WALL      1  
7        LVAC.WALL      2  


Empty cell (unused).